# Islamic Finance & Shariah-Compliant Stock Screening

This notebook provides an educational introduction to:
1. Core principles of Islamic finance
2. AAOIFI screening standards
3. How our screening system works
4. Practical examples of compliance checking

---

## What is Shariah-Compliant Investing?

Islamic finance is based on principles derived from the Quran and Sunnah. The key prohibitions relevant to investing include:

### 1. **Riba (Interest/Usury)**
- Interest-based transactions are prohibited
- This excludes conventional banks, insurance companies, and heavily leveraged companies

### 2. **Gharar (Excessive Uncertainty)**
- Transactions with excessive uncertainty or speculation are prohibited
- This affects certain derivatives and speculative instruments

### 3. **Haram (Prohibited) Activities**
- Alcohol, tobacco, pork, gambling, adult entertainment, weapons
- Companies deriving significant revenue from these are excluded

---

## AAOIFI Screening Standards

The **Accounting and Auditing Organization for Islamic Financial Institutions (AAOIFI)** sets standards used globally. Our system implements AAOIFI Standard No. 21.

### Financial Ratio Screens

| Ratio | Threshold | Purpose |
|-------|-----------|----------|
| Interest-Bearing Debt / Market Cap | < 30% | Limits exposure to riba |
| Interest-Bearing Deposits / Market Cap | < 30% | Limits interest income potential |
| Non-Permissible Income / Total Revenue | < 5% | Limits prohibited income |
| Accounts Receivable / Market Cap | < 49% | Limits debt-trading concerns |

### Why Market Capitalization?

Using market cap as denominator (vs total assets) means:
- Ratios fluctuate with stock price
- A stock could become non-compliant due to price drops
- Regular re-screening is necessary

In [ ]:
# Setup: Add parent directory to path for imports
import sys
sys.path.insert(0, '..')

# Import our screening modules
from shariah.business_screener import BusinessScreener
from shariah.financial_screener import FinancialScreener, FinancialData
from shariah.compliance_engine import ComplianceEngine
from config.shariah_config import shariah_config

print("Modules loaded successfully!")

---

## Business Activity Screening

The first step is to check if a company's primary business is permissible.

### NAICS/SIC Code Screening

We use industry classification codes to identify prohibited businesses:

In [ ]:
# Initialize the business screener
business_screener = BusinessScreener()

# Example 1: Technology company (permissible)
result = business_screener.screen(
    symbol="AAPL",
    industry_name="Consumer Electronics",
    sector="Technology"
)

print(f"AAPL Screening Result:")
print(f"  Compliant: {result.is_compliant}")
print(f"  Status: {result.status}")
print(f"  Reasons: {result.reasons}")

In [ ]:
# Example 2: Bank (non-compliant - conventional finance)
result = business_screener.screen(
    symbol="JPM",
    naics_code="522110",  # Commercial Banking
    industry_name="Commercial Banking",
    sector="Financial Services"
)

print(f"JPM Screening Result:")
print(f"  Compliant: {result.is_compliant}")
print(f"  Status: {result.status}")
print(f"  Prohibited Industries: {result.prohibited_industries}")
print(f"  Reasons: {result.reasons}")

In [ ]:
# Example 3: Alcohol company (non-compliant)
result = business_screener.screen(
    symbol="BUD",
    naics_code="312120",  # Breweries
    industry_name="Beverages - Brewers",
    sector="Consumer Staples"
)

print(f"BUD Screening Result:")
print(f"  Compliant: {result.is_compliant}")
print(f"  Prohibited Industries: {result.prohibited_industries}")
print(f"  Reasons: {result.reasons}")

---

## Financial Ratio Screening

Even if a company's business is permissible, its financial structure must comply with AAOIFI ratios.

In [ ]:
# Initialize financial screener
financial_screener = FinancialScreener()

# Print current thresholds
print("AAOIFI Thresholds:")
print(f"  Max Debt/Market Cap: {shariah_config.max_debt_to_market_cap:.0%}")
print(f"  Max Deposits/Market Cap: {shariah_config.max_deposits_to_market_cap:.0%}")
print(f"  Max Impermissible Income: {shariah_config.max_impermissible_income_ratio:.0%}")
print(f"  Max Receivables/Market Cap: {shariah_config.max_receivables_to_market_cap:.0%}")

In [ ]:
# Example: Compliant company (low debt, low interest income)
compliant_data = FinancialData(
    symbol="TECH_CO",
    market_cap=100_000_000_000,  # $100B market cap
    total_debt=15_000_000_000,   # $15B debt (15%)
    cash_and_equivalents=20_000_000_000,  # $20B cash (20%)
    accounts_receivable=10_000_000_000,   # $10B receivables (10%)
    total_revenue=50_000_000_000,  # $50B revenue
    interest_income=500_000_000,   # $500M interest income (1%)
)

result = financial_screener.screen(compliant_data)

print(f"TECH_CO Financial Screening:")
print(f"  Compliant: {result.is_compliant}")
print(f"  Status: {result.status}")
print(f"\nRatios:")
print(f"  Debt/Market Cap: {result.ratios.debt_to_market_cap:.1%} (limit: 30%)")
print(f"  Deposits/Market Cap: {result.ratios.deposits_to_market_cap:.1%} (limit: 30%)")
print(f"  Impermissible Income: {result.ratios.impermissible_income_ratio:.1%} (limit: 5%)")

In [ ]:
# Example: Non-compliant company (high debt)
non_compliant_data = FinancialData(
    symbol="LEVERED_CO",
    market_cap=50_000_000_000,   # $50B market cap
    total_debt=25_000_000_000,   # $25B debt (50%!) - exceeds 30%
    total_revenue=30_000_000_000,
    interest_income=300_000_000,  # 1% - within limit
)

result = financial_screener.screen(non_compliant_data)

print(f"LEVERED_CO Financial Screening:")
print(f"  Compliant: {result.is_compliant}")
print(f"  Status: {result.status}")
print(f"\nRatios:")
print(f"  Debt/Market Cap: {result.ratios.debt_to_market_cap:.1%} (limit: 30%)")
print(f"\nThresholds Exceeded: {result.thresholds_exceeded}")
print(f"Reasons: {result.reasons}")

---

## Dividend Purification

When investing in Shariah-compliant companies, a small portion of income may still come from non-permissible sources (within the 5% threshold). This portion must be "purified" through charitable donation.

### Purification Formula

```
Purification Amount = Dividend × (Non-Compliant Income / Total Income)
```

### Example

- Company ABC has 3% non-permissible income (within 5% threshold, so compliant)
- You receive $1,000 in dividends
- Purification amount: $1,000 × 0.03 = **$30 to donate to charity**

In [ ]:
from shariah.purification import PurificationCalculator
from decimal import Decimal

# Initialize calculator
calc = PurificationCalculator()

# Set purification ratios for stocks (from their financial statements)
calc.set_purification_ratio("AAPL", 0.01)  # 1% non-compliant income
calc.set_purification_ratio("MSFT", 0.02)  # 2% non-compliant income

# Calculate purification for dividends received
record1 = calc.calculate_dividend_purification(
    symbol="AAPL",
    dividend_amount=500.00,
    shares_held=100
)

record2 = calc.calculate_dividend_purification(
    symbol="MSFT",
    dividend_amount=300.00,
    shares_held=50
)

print("Purification Records:")
print(f"\nAAPL Dividend: ${record1.gross_amount}")
print(f"  Purification ratio: {record1.purification_ratio:.1%}")
print(f"  Amount to purify: ${record1.purification_amount}")

print(f"\nMSFT Dividend: ${record2.gross_amount}")
print(f"  Purification ratio: {record2.purification_ratio:.1%}")
print(f"  Amount to purify: ${record2.purification_amount}")

print(f"\nTotal outstanding purification: ${calc.get_outstanding_balance()}")

---

## Using the Compliance Engine

The `ComplianceEngine` combines all screening methods for a unified compliance check:

In [ ]:
# Initialize the compliance engine
engine = ComplianceEngine()

# Screen a company with full data
result = engine.screen(
    symbol="NVDA",
    business_data={
        "industry_name": "Semiconductors",
        "sector": "Technology",
    },
    financial_data={
        "market_cap": 1_200_000_000_000,
        "total_debt": 10_000_000_000,
        "cash": 20_000_000_000,
        "revenue": 60_000_000_000,
        "interest_income": 400_000_000,
    },
    screening_level="FULL"
)

print(f"NVDA Compliance Result:")
print(f"  Compliant: {result.is_compliant}")
print(f"  Status: {result.status}")
print(f"  Source: {result.source}")
print(f"  Confidence: {result.confidence}")
print(f"  Reasons: {result.reasons}")

---

## Shariah Index Integration

The easiest approach is to use pre-vetted stocks from established Shariah indices:

- **S&P 500 Shariah Index** (via SPUS ETF)
- **Dow Jones Islamic Market Index**
- **MSCI World Islamic Index**

These indices are maintained by professional Shariah scholars who perform rigorous quarterly reviews.

In [ ]:
import asyncio
from shariah.index_integration import ShariahIndexIntegration

# Initialize index integration
index = ShariahIndexIntegration()

# Fetch SPUS holdings (S&P 500 Shariah)
async def load_index():
    data = await index.fetch_etf_holdings("SPUS")
    return data

# Run async function
spus_data = asyncio.get_event_loop().run_until_complete(load_index())

if spus_data:
    print(f"S&P 500 Shariah Index:")
    print(f"  Total constituents: {spus_data.total_count}")
    print(f"  Last updated: {spus_data.last_updated}")
    print(f"\nTop 10 Holdings:")
    for i, holding in enumerate(index.get_top_holdings(10)):
        print(f"  {i+1}. {holding.symbol}: {holding.name} ({holding.weight:.1f}%)")

---

## Summary

### Key Takeaways:

1. **Business Screening**: Exclude companies with prohibited primary activities

2. **Financial Screening**: Apply AAOIFI ratio thresholds
   - Debt < 30% of market cap
   - Interest-bearing deposits < 30% of market cap
   - Non-permissible income < 5% of revenue

3. **Purification**: Donate the non-permissible portion of dividends

4. **Regular Review**: Compliance can change - rescreen quarterly

5. **Index Approach**: Using established Shariah indices provides convenience and scholarly oversight

### Next Steps:

- Explore the ML model notebooks for signal generation
- Review the backtesting framework
- Set up paper trading with IBKR